# 01 — Exploratory Data Analysis: PlantVillage Dataset

**Dataset**: `mohanty/PlantVillage` on HuggingFace Hub  
**Paper**: Mohanty et al. (2016) — *Using Deep Learning for Image-Based Plant Disease Detection*

This notebook covers:
1. Dataset loading via HuggingFace
2. Class distribution visualisation
3. Sample image grid per class
4. Colour vs segmented image comparison
5. Crop-level and disease-level breakdowns

In [ ]:
# Install dependencies if running in Colab / fresh environment
# !pip install -q datasets torch torchvision timm opencv-python matplotlib seaborn tqdm

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from PIL import Image
from datasets import load_dataset

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

print('Libraries loaded.')

## 1. Load the dataset (all three configurations)

In [ ]:
# Load colour configuration (RGB images — default)
ds_color = load_dataset('mohanty/PlantVillage', 'color')
print('Color:', ds_color)

# Load segmented configuration (background removed)
ds_seg   = load_dataset('mohanty/PlantVillage', 'segmented')
print('Segmented:', ds_seg)

In [ ]:
# Inspect a single sample
sample = ds_color['train'][0]
print('Keys    :', list(sample.keys()))
print('Image   :', sample['image'].size, sample['image'].mode)
print('Label   :', sample['label'], '→', ds_color['train'].features['label'].int2str(sample['label']))
print('Crop    :', sample['crop'])
print('Disease :', sample['disease'])
print('leaf_id :', sample['leaf_id'])

## 2. Class distribution

In [ ]:
train_split = ds_color['train']
class_names = train_split.features['label'].names
num_classes = len(class_names)

# Count labels across train + test
all_splits  = [ds_color['train'], ds_color['test']]
label_counts = Counter()
for split in all_splits:
    label_counts.update(split['label'])

labels_sorted = sorted(class_names, key=lambda c: -label_counts[class_names.index(c)])
counts_sorted = [label_counts[class_names.index(c)] for c in labels_sorted]

print(f'Total classes  : {num_classes}')
print(f'Total images   : {sum(label_counts.values()):,}')
print(f'Most common    : {labels_sorted[0]} ({counts_sorted[0]:,} imgs)')
print(f'Least common   : {labels_sorted[-1]} ({counts_sorted[-1]:,} imgs)')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
colors  = plt.cm.tab20(np.linspace(0, 1, num_classes))
bars    = ax.barh(labels_sorted, counts_sorted, color=colors)

ax.set_xlabel('Number of images', fontsize=12)
ax.set_title('PlantVillage — Class Distribution (train + test)', fontsize=13, fontweight='bold')
ax.bar_label(bars, padding=3, fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

## 3. Crop-level breakdown

In [ ]:
crop_counter = Counter(train_split['crop'])
crops_sorted = sorted(crop_counter, key=lambda c: -crop_counter[c])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(crops_sorted, [crop_counter[c] for c in crops_sorted], color='steelblue')
ax.set_xlabel('Crop')
ax.set_ylabel('Training images')
ax.set_title('Training images per crop species', fontsize=12, fontweight='bold')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print('Unique crops:', len(crop_counter))
for crop, cnt in sorted(crop_counter.items(), key=lambda x: -x[1]):
    print(f'  {crop:<20} {cnt:>6,}')

## 4. Sample image grid (5 classes × 4 images)

In [ ]:
N_CLASSES = 8
N_PER_CLS = 4

# Collect samples per class
samples_by_class = {i: [] for i in range(N_CLASSES)}
for sample in train_split:
    lbl = sample['label']
    if lbl < N_CLASSES and len(samples_by_class[lbl]) < N_PER_CLS:
        samples_by_class[lbl].append(sample['image'])
    if all(len(v) == N_PER_CLS for v in samples_by_class.values()):
        break

fig, axes = plt.subplots(N_CLASSES, N_PER_CLS, figsize=(N_PER_CLS * 3, N_CLASSES * 2.5))

for row in range(N_CLASSES):
    for col in range(N_PER_CLS):
        ax = axes[row][col]
        imgs = samples_by_class.get(row, [])
        if col < len(imgs):
            ax.imshow(imgs[col])
        ax.axis('off')
        if col == 0:
            short = class_names[row].replace('___', '\n')
            ax.set_ylabel(short, fontsize=8, rotation=0, labelpad=90, va='center')

fig.suptitle('Sample Images per Class (first 8 classes)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_grid.png', bbox_inches='tight')
plt.show()

## 5. Colour vs Segmented comparison

In [ ]:
N_EXAMPLES = 6

# Pull matching colour and segmented samples (same index = same leaf)
color_samples = [ds_color['train'][i]['image'] for i in range(N_EXAMPLES)]
seg_samples   = [ds_seg['train'][i]['image']   for i in range(N_EXAMPLES)]
labels_ex     = [class_names[ds_color['train'][i]['label']] for i in range(N_EXAMPLES)]

fig, axes = plt.subplots(2, N_EXAMPLES, figsize=(N_EXAMPLES * 3, 6))

for col in range(N_EXAMPLES):
    axes[0][col].imshow(color_samples[col])
    axes[0][col].axis('off')
    axes[0][col].set_title(labels_ex[col].split('___')[0], fontsize=8)

    axes[1][col].imshow(seg_samples[col])
    axes[1][col].axis('off')

axes[0][0].set_ylabel('Color',     fontsize=10, rotation=90)
axes[1][0].set_ylabel('Segmented', fontsize=10, rotation=90)

fig.suptitle('Colour vs Pre-Segmented Images', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('color_vs_segmented.png', bbox_inches='tight')
plt.show()

## 6. Image size distribution

In [ ]:
# Sample 500 images to check size distribution
N_SAMPLE = 500
indices  = np.random.choice(len(train_split), N_SAMPLE, replace=False)
widths, heights = [], []

for i in indices:
    img = train_split[int(i)]['image']
    widths.append(img.width)
    heights.append(img.height)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(widths,  bins=20, color='steelblue', edgecolor='white')
ax1.set_title('Width distribution (px)')
ax1.set_xlabel('Pixels')

ax2.hist(heights, bins=20, color='coral', edgecolor='white')
ax2.set_title('Height distribution (px)')
ax2.set_xlabel('Pixels')

plt.suptitle(f'Image Size Distribution (n={N_SAMPLE})', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Width  — mean: {np.mean(widths):.0f}  min: {min(widths)}  max: {max(widths)}')
print(f'Height — mean: {np.mean(heights):.0f}  min: {min(heights)}  max: {max(heights)}')

## 7. Leaf ID grouping — data leakage analysis

The dataset groups multiple images of the **same physical leaf** under a shared `leaf_id`.  
The pre-built 80/20 split ensures no leaf appears in both train and test — preventing data leakage.

In [ ]:
train_leaf_ids = set(ds_color['train']['leaf_id'])
test_leaf_ids  = set(ds_color['test']['leaf_id'])
overlap        = train_leaf_ids & test_leaf_ids

print(f'Unique leaf IDs in train : {len(train_leaf_ids):,}')
print(f'Unique leaf IDs in test  : {len(test_leaf_ids):,}')
print(f'Overlap (leakage)        : {len(overlap):,}  ← should be 0')

# Images per leaf
from collections import Counter
imgs_per_leaf = Counter(ds_color['train']['leaf_id'])
counts_per_leaf = list(imgs_per_leaf.values())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(counts_per_leaf, bins=30, color='purple', edgecolor='white')
ax.set_title('Images per leaf (train split)', fontsize=11)
ax.set_xlabel('Images per physical leaf')
ax.set_ylabel('Number of leaves')
plt.tight_layout()
plt.show()

print(f'\nMean images per leaf: {np.mean(counts_per_leaf):.1f}')
print(f'Max  images per leaf: {max(counts_per_leaf)}')

---
## Summary

| Metric | Value |
|--------|-------|
| Total images | 54,306 |
| Crop species | 14 |
| Diseases | 26 |
| Classes (species × condition) | 38 |
| Train / Test | 43,596 / 10,709 |
| Configs | color, grayscale, segmented |
| Leakage between splits | None (leaf_id-aware split) |

**Next**: `02_classification.ipynb` — transfer learning with EfficientNet-B0.